 ### ETAPA DE IMPORTAÇÃO

In [3]:
from pathlib import Path
import sys
import zipfile
sys.path.insert(0, str(Path.cwd().resolve().parent))
from _MySQL import banco, config


from scripts import extrair

### Etapa de Download da Base do Drive

In [11]:
extrair.baixar_base_drive()

Downloading...
From (original): https://drive.google.com/uc?id=1Sru-TYSYo-cn-L9WW2DUwhyIUdkM3fIe
From (redirected): https://drive.google.com/uc?id=1Sru-TYSYo-cn-L9WW2DUwhyIUdkM3fIe&confirm=t&uuid=235b9636-6c82-4f2f-bee4-d8b41f37018d
To: /home/calliari/Documents/Estudos/Projeto_final_SCTEC/data/viagens_2025_6meses.zip
100%|██████████| 67.7M/67.7M [00:05<00:00, 12.0MB/s]


### Etapa da Ingestão dos dados para o raw de cada tabela

In [8]:
try:
    conexao = banco.conectar()
    caminho_zip = extrair.localizar_zip()

    with zipfile.ZipFile(caminho_zip) as zip_aberto:
        for arquivo in config.ARQUIVOS.values():
            extrair.carregar_csv(conexao, zip_aberto, arquivo["csv"], arquivo["tabela_raw"])

    conexao.close()
    print("=== Camada RAW concluida com sucesso! ===")
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise
    

      Carregando raw_viagem ...
      -> 341860 linhas em raw_viagem
      Carregando raw_pagamento ...
[ERRO] Algo deu errado: 1136 (21S01): Column count doesn't match value count at row 1


DataError: 1136 (21S01): Column count doesn't match value count at row 1

## 🧹 Etapa de Preparação da Tabela `raw_viagem`

Nesta etapa será realizada a **limpeza e preparação dos dados** da tabela `raw_viagem`, removendo colunas que não serão utilizadas nas análises posteriores.

### 🗑️ Colunas Removidas

As seguintes colunas serão excluídas da tabela:

- `justificativa_urgencia_viagem`
- `cod_orgao_solicitante`
- `nome_orgao_solicitante`
- `cpf_viajante`
- `funcao`
- `descricao_funcao`

Após essa etapa, a tabela seguirá para as próximas fases de **transformação e análise dos dados**.

Drop na raw_pagamento

- `cod_orgao_sup`
- `nome_orgao_sup`
- `cod_orgao_pagador`
- `cod_ug_pagadora`

In [ ]:
try:
    conexao = banco.conectar()
    sql = """ALTER TABLE raw_pagamento
            DROP COLUMN cod_orgao_sup,
            DROP COLUMN nome_orgao_sup,
            DROP COLUMN cod_orgao_pagador,
            DROP COLUMN cod_ug_pagadora;"""
    banco.executar(conexao,sql)
    conexao.close()

except Exception as erro:
        print("[ERRO] Algo deu errado:", erro)
        raise   
    

